# Updated Games Today/Tomorrow (winner-focused)

Refined daily notebook for outright winner selection.

- Uses same V6-style feature structure and tunable compression knobs.
- Trains probability calibration on **completed games in the current season parquet**.
- Scores **today/tomorrow** slate games and outputs ranked winner picks.
- Kalshi columns are optional; workflow still runs if they are missing.

In [22]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 240)

SEASON = datetime.now().year
LOOKAHEAD_DAYS = 1   # 0=today only, 1=today+tomorrow
TOP_N_PER_DATE = 3

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

# Tunable compression / penalty knobs (aligned with updated backtests)
RISK_W_CONFLICT = 0.30
RISK_W_INSTABILITY = 0.40
RISK_W_SIGNAL_GAP = 0.30
RISK_TANH_ALPHA = 0.80
QUALITY_INSTABILITY_DECAY = 0.90
TRAP_PENALTY_WEIGHT = 0.35

SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

# Leg weights (aligned with updated backtests): less platoon / damped pen / softer coherence mult
OFF_W_OFFENSE = 0.85
OFF_W_PLATOON = 0.15
PEN_LEG_DAMP = 0.65
COHERENCE_SOFT_MIN = 0.50
COHERENCE_SOFT_RANGE = 0.50

print({"SEASON": SEASON, "LOOKAHEAD_DAYS": LOOKAHEAD_DAYS, "TOP_N_PER_DATE": TOP_N_PER_DATE})

{'SEASON': 2026, 'LOOKAHEAD_DAYS': 1, 'TOP_N_PER_DATE': 3}


In [23]:
cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"
if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run pipeline for season {SEASON}.")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

# Train set: completed games with outcomes
train = df[df["detailed_state"].isin(FINAL_STATES)].copy()
if "home_win" in train.columns:
    train = train[train["home_win"].notna()].copy()

# Scoring slate: today/tomorrow (or wider if LOOKAHEAD_DAYS changed)
today = pd.Timestamp.today().normalize()
end_day = today + pd.Timedelta(days=LOOKAHEAD_DAYS)
slate = df[df["game_date"].between(today, end_day)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)} | completed(train)={len(train)} | slate={len(slate)}")
print(f"Slate dates: {today.date()} .. {end_day.date()}")

Using: /Users/austinlolli/Code/baseball/mlb-model/data/games_2026.parquet
Rows total=2796 | completed(train)=617 | slate=25
Slate dates: 2026-04-16 .. 2026-04-17


In [24]:
def build_v6_scores(frame: pd.DataFrame) -> pd.DataFrame:
    s = frame.copy()

    # Missing SP/team legs would otherwise NaN-out the whole score. Treat as neutral (0) so
    # calibration and daily picks stay finite (same idea as neutral pen_norm fill).
    kbb = pd.to_numeric(s["sp_kbb_diff"], errors="coerce").fillna(0.0)
    xfi = pd.to_numeric(s["sp_xfip_diff"], errors="coerce").fillna(0.0)
    off = pd.to_numeric(s["offense_diff"], errors="coerce").fillna(0.0)

    kbb_norm = kbb / SP_KBB_ABS
    xfip_norm = -xfi / SP_XFIP_ABS
    off_norm = off / OFFENSE_ABS
    platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

    if "home_pen_roll14_fip" in s.columns:
        hr_pen = s["home_pen_roll14_fip"]
        if "home_pen_season_fip" in s.columns:
            hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
    else:
        hr_pen = pd.Series(np.nan, index=s.index)

    if "away_pen_roll14_fip" in s.columns:
        ar_pen = s["away_pen_roll14_fip"]
        if "away_pen_season_fip" in s.columns:
            ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
    else:
        ar_pen = pd.Series(np.nan, index=s.index)

    pen_gap = hr_pen - ar_pen  # home - away; lower home FIP better (aligned with sp_xfip_diff)
    pen_norm = (-pen_gap / PEN_ABS).fillna(0.0)

    sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
    off_vector = (OFF_W_OFFENSE * off_norm) + (OFF_W_PLATOON * platoon_norm)
    pen_vector = PEN_LEG_DAMP * pen_norm

    signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
    sign_matrix = np.sign(signal_matrix)
    mag_matrix = np.abs(signal_matrix)

    mean_sign = np.mean(sign_matrix, axis=0)
    mean_mag = np.mean(mag_matrix, axis=0)

    # Per-game variance (not a mean across games — that poisoned all rows if any game had NaN signs).
    agreement = 1 - np.nanvar(sign_matrix, axis=0)
    direction_purity = np.abs(mean_sign)
    mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

    coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
    coherence_mult = COHERENCE_SOFT_MIN + COHERENCE_SOFT_RANGE * coherence_score
    raw_edge = mean_mag
    instability = np.std(signal_matrix, axis=0)

    direction_conflict = (
        (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
        + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
        + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
    )

    risk_penalty_raw = (
        RISK_W_CONFLICT * direction_conflict
        + RISK_W_INSTABILITY * instability
        + RISK_W_SIGNAL_GAP * np.abs(sp_vector - off_vector)
    )
    risk_penalty = np.tanh(RISK_TANH_ALPHA * risk_penalty_raw)

    trap_score = raw_edge * (1 - coherence_score)
    quality_score = raw_edge * coherence_mult * np.exp(-QUALITY_INSTABILITY_DECAY * instability) * (1 / (1 + risk_penalty))
    risk_adjusted_edge = quality_score - TRAP_PENALTY_WEIGHT * trap_score

    prefer_home = sp_vector >= 0
    s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
    s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

    s["coherence_score"] = coherence_score
    s["quality_score"] = quality_score
    s["risk_adjusted_edge"] = risk_adjusted_edge
    s["instability"] = instability
    s["risk_penalty_raw"] = risk_penalty_raw
    s["risk_penalty"] = risk_penalty

    if "home_win" in s.columns:
        hw = pd.to_numeric(s["home_win"], errors="coerce")
        s["favorite_won"] = np.where(prefer_home, hw == 1.0, hw == 0.0)
    else:
        s["favorite_won"] = np.nan

    return s


train_scored = build_v6_scores(train)
slate_scored = build_v6_scores(slate)

print(f"train_scored={len(train_scored)} | slate_scored={len(slate_scored)}")

train_scored=617 | slate_scored=25


In [25]:
# Fit calibration on completed games, then project onto today's slate
cal = train_scored.copy()
cal = cal[cal["favorite_won"].isin([True, False])].copy()
cal["y"] = cal["favorite_won"].astype(int)
cal["score_raw"] = pd.to_numeric(cal["risk_adjusted_edge"], errors="coerce")
cal = cal.dropna(subset=["score_raw", "y"]).copy()

if len(cal) == 0:
    raise RuntimeError(
        "0 calibration rows after dropna(score_raw). Usually every row had NaN risk_adjusted_edge "
        "because core diffs were all missing — re-run after the build_v6_scores neutral-fill fix, "
        "or check parquet has sp_kbb_diff / sp_xfip_diff / offense_diff populated."
    )

if len(cal) < 200:
    print(f"Warning: only {len(cal)} calibration rows; probabilities may be noisy.")

x = cal[["score_raw"]].to_numpy()
y = cal["y"].to_numpy()

logit = LogisticRegression(max_iter=2000)
logit.fit(x, y)

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(cal["score_raw"].to_numpy(), y)

slate_scored = slate_scored.copy()
slate_scored["score_raw"] = pd.to_numeric(slate_scored["risk_adjusted_edge"], errors="coerce")
# Keep slate rows even if score_raw is NaN: fall back to rank by quality_score
_bad = slate_scored["score_raw"].isna()
if _bad.any():
    print(f"Warning: {int(_bad.sum())} slate rows have NaN risk_adjusted_edge; filling score_raw from quality_score for ranking only.")
    slate_scored.loc[_bad, "score_raw"] = pd.to_numeric(slate_scored.loc[_bad, "quality_score"], errors="coerce")

slate_scored = slate_scored.dropna(subset=["score_raw"]).copy()

x_slate = slate_scored[["score_raw"]].to_numpy()
slate_scored["p_model_logit"] = logit.predict_proba(x_slate)[:, 1]
slate_scored["p_model_iso"] = iso.predict(slate_scored["score_raw"].to_numpy())

# Primary probability for outright-winner ranking
slate_scored["p_model"] = 0.5 * slate_scored["p_model_logit"] + 0.5 * slate_scored["p_model_iso"]

# Optional market comparison (only if Kalshi implied exists)
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(slate_scored.columns):
    favored_home = slate_scored["_mismatch_side"].eq("home_favored")
    slate_scored["market_implied_favored"] = np.where(
        favored_home,
        slate_scored["kalshi_home_implied"],
        slate_scored["kalshi_away_implied"],
    )
    slate_scored["edge_vs_market"] = slate_scored["p_model"] - slate_scored["market_implied_favored"]
else:
    slate_scored["market_implied_favored"] = np.nan
    slate_scored["edge_vs_market"] = np.nan

print("Calibration/train rows:", len(cal))
print("Slate rows with probabilities:", len(slate_scored))

Calibration/train rows: 617
Slate rows with probabilities: 25


In [26]:
# Daily actionable winner picks: full slate per date, ordered by calibrated probability
picks = slate_scored.copy()
picks = picks.sort_values(["game_date", "p_model"], ascending=[True, False]).copy()
picks["rank_in_slate"] = picks.groupby("game_date")["p_model"].rank(method="first", ascending=False)

# Keep the full slate while preserving rank order for each date.
daily_top = picks.copy()

# Confidence tiers for practical use
# (Can be tuned after more backtests)
daily_top["confidence_tier"] = np.select(
    [
        daily_top["p_model"] >= 0.57,
        daily_top["p_model"] >= 0.54,
        daily_top["p_model"] >= 0.51,
    ],
    ["A", "B", "C"],
    default="D",
)

display_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side",
    "rank_in_slate", "confidence_tier",
    "p_model", "p_model_logit", "p_model_iso",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "market_implied_favored", "edge_vs_market",
    "home_probable_pitcher", "away_probable_pitcher",
]
display_cols = [c for c in display_cols if c in daily_top.columns]

print(f"Full slate picks across dates: {len(daily_top)}")
display(daily_top[display_cols])

print("\nBy-date pick counts")
display(daily_top.groupby("game_date").size().rename("pick_count").reset_index())

Full slate picks across dates: 25


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,rank_in_slate,confidence_tier,p_model,p_model_logit,p_model_iso,risk_adjusted_edge,quality_score,coherence_score,instability,market_implied_favored,edge_vs_market,home_probable_pitcher,away_probable_pitcher
641,2026-04-16,Detroit Tigers,Kansas City Royals,Detroit Tigers,home_favored,1.0,A,0.628638,0.564969,0.692308,0.550011,0.582077,0.924909,0.429317,0.305,0.323638,Keider Montero,Kris Bubic
648,2026-04-16,San Diego Padres,Seattle Mariners,Seattle Mariners,away_favored,2.0,B,0.546701,0.542676,0.550725,-0.083162,0.118969,0.279208,0.934867,0.515,0.031701,Walker Buehler,Luis Castillo
640,2026-04-16,Cincinnati Reds,San Francisco Giants,San Francisco Giants,away_favored,3.0,B,0.546474,0.542223,0.550725,-0.095989,0.104854,0.272508,1.040679,0.995,-0.448526,Chase Burns,Landen Roupp
642,2026-04-16,New York Yankees,Los Angeles Angels,Los Angeles Angels,away_favored,4.0,B,0.545529,0.540333,0.550725,-0.149411,0.114388,0.260690,1.258551,0.995,-0.449471,Max Fried,Brent Suter
646,2026-04-16,Cleveland Guardians,Baltimore Orioles,Cleveland Guardians,home_favored,5.0,B,0.545438,0.540152,0.550725,-0.154512,0.127213,0.264884,1.176107,0.545,0.000438,Parker Messick,Shane Baz
643,2026-04-16,Milwaukee Brewers,Toronto Blue Jays,Toronto Blue Jays,away_favored,6.0,B,0.544525,0.538325,0.550725,-0.206102,0.104450,0.252301,1.443772,0.005,0.539525,Brandon Sproat,Patrick Corbin
644,2026-04-16,Chicago White Sox,Tampa Bay Rays,Tampa Bay Rays,away_favored,7.0,D,0.496595,0.532583,0.460606,-0.368091,0.134039,0.241204,1.741122,0.995,-0.498405,Jordan Leasure,Steven Matz
647,2026-04-16,Houston Astros,Colorado Rockies,Houston Astros,home_favored,8.0,D,0.496455,0.532305,0.460606,-0.375941,0.131783,0.240535,1.761378,0.605,-0.108545,Ryan Weiss,Juan Mejia
639,2026-04-16,Pittsburgh Pirates,Washington Nationals,Pittsburgh Pirates,home_favored,9.0,D,0.495890,0.531175,0.460606,-0.407797,0.164070,0.246236,1.597780,0.005,0.490890,Braxton Ashcraft,Foster Griffin
645,2026-04-16,Athletics,Texas Rangers,Texas Rangers,away_favored,10.0,D,0.494994,0.529381,0.460606,-0.458320,0.056048,0.219617,2.591101,0.855,-0.360006,Jacob Lopez,Jack Leiter



By-date pick counts


,game_date,pick_count
0,2026-04-16,10
1,2026-04-17,15


## How to use this daily

- **Primary rank target**: `p_model` (calibrated outright winner probability).
- **Primary selection**: full slate, sorted by `rank_in_slate` / `p_model` within each date.
- **Tie-breakers**: prefer higher `coherence_score`, lower `instability`, and positive `edge_vs_market` (when available).
- **Kalshi note**: if Kalshi columns are missing, picks still work; market edge fields remain `NaN`.

### Metric definitions

- **Brier score**: average squared error of predicted probabilities vs actual outcomes. Lower is better.
- **Log loss**: harsher penalty for confident wrong predictions. Lower is better.
- **AUC**: ranking quality (`0.5` random, `1.0` perfect).
- **Calibration**: how close predicted probabilities are to observed win rates.